In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE SoftmaxCausalBwd BITEXACT TEST (PURE TKV, NO C INJECTION) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so + .tkv C .so. NO C injection. CUNG flags.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/softmax_causal_bwd_pure.cpu.c'; so=ex+'/softmax_causal_bwd_pure.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_softmax_cb' in txt, '(expect False = pure TKV)')
print('f64 dot present =', 'double d_dotv' in txt)
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare SoftmaxCausalBwd
import ctypes, numpy as np
PP=ctypes.POINTER(ctypes.c_float); II=ctypes.c_int
args=[PP,II,PP,II,PP,II,PP,II,II,II]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').SoftmaxCausalBwd
native.argtypes=args; native.restype=II
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/softmax_causal_bwd_pure.so').SoftmaxCausalBwd
tkv.argtypes=args; tkv.restype=II
rng=np.random.default_rng(31)
for n,hd in [(8,16),(16,64),(64,128),(127,64)]:
    dp0=rng.standard_normal(n*n).astype(np.float32)
    P=rng.random((n,n)).astype(np.float32)
    dctx=rng.standard_normal(n*hd).astype(np.float32)
    ctx=rng.standard_normal(n*hd).astype(np.float32)
    a=dp0.copy(); c=dp0.copy()
    native(a.ctypes.data_as(PP),a.size,P.ctypes.data_as(PP),P.size,dctx.ctypes.data_as(PP),dctx.size,ctx.ctypes.data_as(PP),ctx.size,n,hd)
    tkv  (c.ctypes.data_as(PP),c.size,P.ctypes.data_as(PP),P.size,dctx.ctypes.data_as(PP),dctx.size,ctx.ctypes.data_as(PP),ctx.size,n,hd)
    d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
    print('n=%d hd=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(n,hd,d==0,d,a.size,np.abs(a-c).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> SoftmaxCausalBwd pure .tkv replaces the former auto-glue C kernel.')